# Embed & Benchmark — pplx-embed-context-v1-0.6b

Initialize the FP8 runtime, run a quick embedding test, then benchmark with transcript-percentile and fill-budget configs.

In [ ]:
import logging
import torch
from modules import PPLXEmbedFP8Runtime, run_benchmark

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

device = "cuda" if torch.cuda.is_available() else "cpu"
runtime = PPLXEmbedFP8Runtime(
    device=device,
    max_seq_len=32768,
    gpu_memory_utilization=0.90,
    padding_tolerance=1.4,
)
print(f"max_batch_tokens: {runtime.max_batch_tokens:,}")
print(f"FP8: {runtime.fp8_enabled}")

## Quick embedding test

In [ ]:
sample_docs = [
    {"doc_id": "doc_1", "text": "First document text here."},
    {"doc_id": "doc_2", "text": "Second document text here."},
]
results = runtime.embed_documents(sample_docs)
for r in results:
    print(f"{r.doc_id}: {len(r.chunk_texts)} chunks, emb shape {r.embeddings.shape}")

## Benchmark 1: Transcript percentile configs

In [ ]:
results = run_benchmark(runtime, runtime.tokenizer)

## Benchmark 2: Fill-budget configs

In [ ]:
import gc
import time
import numpy as np
from modules.benchmark import make_docs
from modules.gpu import _cleanup_gpu


def estimate_docs_to_fill_budget(budget_tokens, tokens_per_doc):
    """Calculate how many docs needed to fill the batch budget."""
    return max(1, int(budget_tokens / max(tokens_per_doc, 1)))


budget = runtime.max_batch_tokens
tokenizer = runtime.tokenizer

# (n_docs, approx_tokens_per_doc, description)
# n_docs=None means auto-calculate to fill budget
fill_budget_configs = [
    (None,   128, "Short docs (fill budget)"),
    (None,   512, "Medium docs (fill budget)"),
    (None,  1024, "Long docs (fill budget)"),
    (None,  2048, "Very long docs (fill budget)"),
    (None,  4096, "4K docs (fill budget)"),
    (None,  8192, "8K docs (fill budget)"),
    (None, 32000, "32K docs (fill budget)"),
    (1,    32000, "Single 32K doc"),
    (100,    512, "100 x 512 tok"),
    (50,    1024, "50 x 1K tok"),
    (20,    2048, "20 x 2K tok"),
    "mixed",
]

fb_results = []

for cfg in fill_budget_configs:
    if cfg == "mixed":
        print(f"\n{'─' * 80}")
        print("Mixed workload (realistic, budget-filling)")
        mixed_docs = []
        target_tokens = budget * 2
        accumulated = 0
        lengths = [
            64, 128, 256, 512, 1024, 2048, 4096, 512, 128, 256,
            1024, 64, 512, 128, 2048, 256, 4096, 512, 1024, 128,
        ]
        idx = 0
        while accumulated < target_tokens:
            length = lengths[idx % len(lengths)]
            d, actual = make_docs(1, length, tokenizer)
            d[0]["doc_id"] = f"mixed_{len(mixed_docs)}_{length}"
            mixed_docs.extend(d)
            accumulated += actual
            idx += 1

        print(f"  {len(mixed_docs)} docs, ~{accumulated:,} total tokens "
              f"(target: {budget * 2:,} = 2x budget)")

        _cleanup_gpu()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        runtime.embed_documents(mixed_docs, chunking="semantic", show_progress=False)
        torch.cuda.synchronize()
        elapsed = time.time() - t0
        stats = runtime.get_stats()
        peak = torch.cuda.max_memory_allocated() / 1e9

        print(f"  {stats.n_batches} batches, {stats.oom_retries} OOM")
        print(f"  {elapsed:.2f}s | {stats.tokens_per_sec:,.0f} tok/s | {stats.chunks_per_sec:,.0f} ch/s")
        print(f"  GPU peak: {peak:.2f} GB")

        fb_results.append({
            "desc": f"Mixed ({len(mixed_docs)} docs)",
            "n_docs": len(mixed_docs), "tokens": stats.n_tokens,
            "chunks": stats.n_chunks, "batches": stats.n_batches,
            "elapsed": elapsed, "tok_per_s": stats.tokens_per_sec,
            "chunk_per_s": stats.chunks_per_sec, "peak_gb": peak,
        })
        continue

    n_docs, approx_tok, desc = cfg

    if n_docs is None:
        _, actual_tok = make_docs(1, approx_tok, tokenizer)
        n_docs = estimate_docs_to_fill_budget(budget, actual_tok)
        n_docs = max(n_docs, 1)

    docs, actual_tok = make_docs(n_docs, approx_tok, tokenizer)
    total_expected = n_docs * actual_tok

    print(f"\n{'─' * 80}")
    print(f"{desc}: {n_docs} docs x ~{actual_tok} tok = ~{total_expected:,} total "
          f"(budget: {budget:,})")

    _cleanup_gpu()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    runtime.embed_documents(docs, chunking="semantic", show_progress=False)
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    stats = runtime.get_stats()
    peak = torch.cuda.max_memory_allocated() / 1e9

    print(f"  {stats.n_batches} batches, {stats.oom_retries} OOM")
    print(f"  {elapsed:.2f}s | {stats.tokens_per_sec:,.0f} tok/s | {stats.chunks_per_sec:,.0f} ch/s")
    print(f"  GPU peak: {peak:.2f} GB")

    fb_results.append({
        "desc": desc, "n_docs": n_docs, "tokens": stats.n_tokens,
        "chunks": stats.n_chunks, "batches": stats.n_batches,
        "elapsed": elapsed, "tok_per_s": stats.tokens_per_sec,
        "chunk_per_s": stats.chunks_per_sec, "peak_gb": peak,
    })

# Summary
print(f"\n{'=' * 80}")
print("FILL-BUDGET SUMMARY")
print(f"{'=' * 80}")
print(f"{'Workload':<30} {'Docs':>6} {'Tokens':>10} {'Chunks':>7} "
      f"{'Batch':>6} {'Time':>7} {'Tok/s':>10} {'Ch/s':>8} {'Peak':>6}")
print("─" * 102)
for r in fb_results:
    print(f"{r['desc']:<30} {r['n_docs']:>6} {r['tokens']:>10,} "
          f"{r['chunks']:>7} {r['batches']:>6} {r['elapsed']:>6.2f}s "
          f"{r['tok_per_s']:>10,.0f} {r['chunk_per_s']:>8,.0f} "
          f"{r['peak_gb']:>5.1f}G")

best = max(fb_results, key=lambda r: r["tok_per_s"])
print(f"\nPeak: {best['tok_per_s']:,.0f} tok/s on '{best['desc']}'")

## GPU Memory Summary

In [ ]:
if torch.cuda.is_available():
    current = torch.cuda.memory_allocated() / 1e9
    peak = torch.cuda.max_memory_allocated() / 1e9
    _, total = torch.cuda.mem_get_info()
    print(f"GPU memory: {current:.2f} GB current, {peak:.2f} GB peak of {total/1e9:.0f} GB total")
else:
    print("No GPU available")